In [1]:
from hu_news_scrape import extract_article_data, filter_links, get_multiple_site_links, get_relevant_links_of_site
import pandas as pd
import requests
from extractor import NewsExtractor

In [2]:
print("=== Help: extract_article_data ===")
help(extract_article_data)

print("\n=== Help: filter_links ===")
help(filter_links)

print("\n=== Help: get_multiple_site_links ===")
help(get_multiple_site_links)

print("\n=== Help: get_relevant_links_of_site ===")
help(get_relevant_links_of_site)

=== Help: extract_article_data ===
Help on function extract_article_data in module hu_news_scrape:

extract_article_data(url, only_today=False)


=== Help: filter_links ===
Help on function filter_links in module hu_news_scrape:

filter_links(soup, prefix)
    Szűri a valószínű hírcikk linkeket különböző hírportálokról.


=== Help: get_multiple_site_links ===
Help on function get_multiple_site_links in module hu_news_scrape:

get_multiple_site_links(site_lst)


=== Help: get_relevant_links_of_site ===
Help on function get_relevant_links_of_site in module hu_news_scrape:

get_relevant_links_of_site(site_url)



In [3]:
sites = [
                                        'https://www.telex.hu'
                                        ,'https://www.blikk.hu'
                                        ,'https://www.origo.hu'
                                        ,'https://444.hu'
                                        ,'https://24.hu'
                                        ,'https://mandiner.hu'
                                        ,'https://index.hu'
                                        #,'https://www.portfolio.hu'
                                        ,'https://borsonline.hu'
                                        ,'https://hvg.hu'
                                        #,'https://kiskegyed.hu'
                                        ,'https://papageno.hu'
                                       ]

In [4]:
site_links = get_multiple_site_links(sites)
len(site_links)

996

In [5]:
import torch
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [6]:
def extract_with_extractor(url):
    response = requests.get(url, timeout=10)
    extractor = NewsExtractor(response.text, url)
    return extractor.extract(with_kw=True)

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time, random

In [8]:
MAX_WORKERS = 12
DELAY_RANGE = (0.05, 0.2)

def safe_extract(url):
    try:
        # tiny random delay is polite, especially if many links are same domain
        time.sleep(random.uniform(*DELAY_RANGE))
        return extract_with_extractor(url)
    except Exception as e:
        # return something structured so downstream code can see failures
        return {"url": url, "error": str(e)}

# preserve original order by storing results by index
def parallel_extract(urls, max_workers=MAX_WORKERS):
    print(f"Starting extraction of URLs with {max_workers} threads.")
    results = [None] * len(urls)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_to_idx = {ex.submit(safe_extract, u): i for i, u in enumerate(urls)}
        for fut in as_completed(future_to_idx):
            i = future_to_idx[fut]
            results[i] = fut.result()
            print(f"⏱️  Progress: {i}")
    print(f"✅ Finished all URLs")
    return results

articles_data = parallel_extract(site_links)

🚀 Starting extraction of URLs with 12 threads.
⏱️  Progress: 9
⏱️  Progress: 4
⏱️  Progress: 2
⏱️  Progress: 10
⏱️  Progress: 8
⏱️  Progress: 0
⏱️  Progress: 13
⏱️  Progress: 1
⏱️  Progress: 3
⏱️  Progress: 6
⏱️  Progress: 11
⏱️  Progress: 7
⏱️  Progress: 12
⏱️  Progress: 16
⏱️  Progress: 15
⏱️  Progress: 5
⏱️  Progress: 20
⏱️  Progress: 14
⏱️  Progress: 18
⏱️  Progress: 22
⏱️  Progress: 19
⏱️  Progress: 17
⏱️  Progress: 21
⏱️  Progress: 25
⏱️  Progress: 23
⏱️  Progress: 30
⏱️  Progress: 24
⏱️  Progress: 27
⏱️  Progress: 26
⏱️  Progress: 28
⏱️  Progress: 32
⏱️  Progress: 29
⏱️  Progress: 33
⏱️  Progress: 31
⏱️  Progress: 34
⏱️  Progress: 44
⏱️  Progress: 43
⏱️  Progress: 36
⏱️  Progress: 38
⏱️  Progress: 40
⏱️  Progress: 37
⏱️  Progress: 39
⏱️  Progress: 41
⏱️  Progress: 42
⏱️  Progress: 35
⏱️  Progress: 46
⏱️  Progress: 45
⏱️  Progress: 50
⏱️  Progress: 47
⏱️  Progress: 48
⏱️  Progress: 53
⏱️  Progress: 51
⏱️  Progress: 52
⏱️  Progress: 54
⏱️  Progress: 55
⏱️  Progress: 59
⏱️  Progres

In [9]:
articles_data[:10]

[{'url': 'https://www.telex.hu/belfold/2025/11/30/karacsony-gergely-fovarosi-onkormanyzat-eloterjesztes-csodhelyzet-budapest',
  'page': 'www.telex.hu',
  'title': 'Karácsony benyújtotta a rendkívüli közgyűlésre az előterjesztését, ami az ÁSZ-jelentésre hivatkozik',
  'text': 'Megjelent a főváros honlapján Karácsony Gergely előterjesztése a hétfői rendkívüli közgyűlésre – vette észre a Magyar Hang. A főpolgármester csütörtökön jelentette be, hogy rendkívüli közgyűlést hívott össze hétfő délutánra a kelenföldi buszgarázsba, miután Gulyás Gergely a kormányinfón arról beszélt, hogy a kormány elkötelezett Budapest működőképességének biztosításában, nyitottak a mentőcsomagra, de a kormány azt várja, hogy a Fővárosi Közgyűlés hivatalosan is állapítsa meg a fizetésképtelenséget.\n\nA közgyűlésre benyújtott dokumentum szerint részben megerősítenének egy korábban, a közgyűlés által már megszavazott javaslatot, aminek lényege, hogy az Állami Számvevőszék egy jelentése szerint a főváros anyagi ne

In [10]:
df = pd.DataFrame(articles_data)

In [11]:
df

,url,page,title,text,author,date,keywords,error
0,https://www.telex.hu/belfold/2025/11/30/karacs...,www.telex.hu,Karácsony benyújtotta a rendkívüli közgyűlésre...,Megjelent a főváros honlapján Karácsony Gergel...,Thüringer Barbara,2025-11-30 00:00:00,"[előterjesztés, Fővárosi Önkormányzat, Belföld...",NaN
1,https://www.telex.hu/after/2025/11/28/sex-pist...,www.telex.hu,Budapestre jön a Sex Pistols frontemberének ze...,"A Sex Pistols egykori frontemberének, Johnny R...",Dévai László,2025-11-28 00:00:00,"[Sex Pistols, Johnny Rotten, Public Image Ltd,...",NaN
2,https://www.telex.hu/pr-cikk/2025/11/28/igy-sz...,www.telex.hu,Így szerezhet most 110 eurótól tulajdonrészt a...,Kihagyhatatlan lehetőséget kínál a kisbefektet...,Ez itt egy PR-cikk,2025-11-28 00:00:00,"[PR2, PR-cikk]",NaN
3,https://www.telex.hu/eszkombajn/2025/10/06/szt...,www.telex.hu,Amikor Sztálin cikkeket írt a Szabad Népbe,"Egy szókimondó, ellentmondást nem tűrő új szer...",Ághassi Attila,2025-10-06 00:00:00,"[Szabad Nép, Észkombájn, Sztálin]",NaN
4,https://www.telex.hu/belfold/2025/11/30/dohany...,www.telex.hu,Lezuhant egy kőkorlát darabja egy Dohány utcai...,Egy olvasónk szerint szombat este a budapesti ...,Thüringer Barbara,2025-11-30 00:00:00,"[Dohány utca, Belföld, volkswagen, rendőrség, ...",NaN
...,...,...,...,...,...,...,...,...
991,https://papageno.hu/tema/blogok/orosz-zenei-fe...,papageno.hu,Orosz Zenei Fesztivál - Papageno,Az orosz zenei tradíció fiatalnak számít az ol...,,None,[],NaN
992,https://papageno.hu/tema/blogok/orgonak-ejszak...,papageno.hu,Orgonák Éjszakája - Papageno,"Olivier Latry, a párizsi Notre-Dame-székesegyh...",None,None,[],NaN
993,https://papageno.hu/magazin/2025/09/oktober-2025/,papageno.hu,Lapozzon bele a Papageno októberi számába!,Megjelent a Papageno magazin októberi száma. A...,Papageno,2025-09-24 10:54:23+00:00,[papageno magazin 2025],NaN
994,https://papageno.hu/tema/blogok/orientale-lumen/,papageno.hu,Orientale Lumen - Papageno,Vajon lesz-e közönsége egy vokális koncertsoro...,None,None,[],NaN


In [12]:
date_df = df[df['date'].notna()]
date_df

,url,page,title,text,author,date,keywords,error
0,https://www.telex.hu/belfold/2025/11/30/karacs...,www.telex.hu,Karácsony benyújtotta a rendkívüli közgyűlésre...,Megjelent a főváros honlapján Karácsony Gergel...,Thüringer Barbara,2025-11-30 00:00:00,"[előterjesztés, Fővárosi Önkormányzat, Belföld...",NaN
1,https://www.telex.hu/after/2025/11/28/sex-pist...,www.telex.hu,Budapestre jön a Sex Pistols frontemberének ze...,"A Sex Pistols egykori frontemberének, Johnny R...",Dévai László,2025-11-28 00:00:00,"[Sex Pistols, Johnny Rotten, Public Image Ltd,...",NaN
2,https://www.telex.hu/pr-cikk/2025/11/28/igy-sz...,www.telex.hu,Így szerezhet most 110 eurótól tulajdonrészt a...,Kihagyhatatlan lehetőséget kínál a kisbefektet...,Ez itt egy PR-cikk,2025-11-28 00:00:00,"[PR2, PR-cikk]",NaN
3,https://www.telex.hu/eszkombajn/2025/10/06/szt...,www.telex.hu,Amikor Sztálin cikkeket írt a Szabad Népbe,"Egy szókimondó, ellentmondást nem tűrő új szer...",Ághassi Attila,2025-10-06 00:00:00,"[Szabad Nép, Észkombájn, Sztálin]",NaN
4,https://www.telex.hu/belfold/2025/11/30/dohany...,www.telex.hu,Lezuhant egy kőkorlát darabja egy Dohány utcai...,Egy olvasónk szerint szombat este a budapesti ...,Thüringer Barbara,2025-11-30 00:00:00,"[Dohány utca, Belföld, volkswagen, rendőrség, ...",NaN
...,...,...,...,...,...,...,...,...
971,https://papageno.hu/blogok/mozart-planet/2025/...,papageno.hu,"Paul Lewis: „Egy zongora akkor a legjobb, ha é...",Paul Lewis a kortárs zenei élet egyik legelism...,Ur Máté,2025-11-28 04:02:25+00:00,"[paul lewis, concerto budapest, beethoven, int...",NaN
979,https://papageno.hu/kiallitas/2025/11/a-magyar...,papageno.hu,A magyar neoavantgárd mestere - Keserü Ilona,Keserü Ilona (1933) a hazai kortárs képzőművés...,Papageno,2025-11-29 04:28:54+00:00,"[képzőművészet, keserü ilona]",NaN
981,https://papageno.hu/promocio/2025/11/a-kekszak...,papageno.hu,Stuttgartban és Bukaresten is sikert aratott A...,2025-ben két alkalommal adta elő Bartók A kéks...,Papageno,2025-11-25 18:29:34+00:00,"[kamara, cser péter, hajdú diána, dénes istván...",NaN
984,https://papageno.hu/magazin/2025/10/lapozzon-b...,papageno.hu,Lapozzon bele a Papageno novemberi számába!,Megjelent a Papageno magazin novemberi száma. ...,Papageno,2025-10-21 10:13:47+00:00,[papageno magazin 2025],NaN


In [14]:
date_df['date'] = pd.to_datetime(date_df['date'], utc=True, errors='coerce').dt.tz_convert(None)
date_df = date_df[date_df['date'].notna()]
date_df

/var/folders/n3/kmxx8m1x08d5n0y3k5mydwym0000gn/T/ipykernel_1242/1387613155.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  date_df['date'] = pd.to_datetime(date_df['date'], utc=True, errors='coerce').dt.tz_convert(None)


,url,page,title,text,author,date,keywords,error
0,https://www.telex.hu/belfold/2025/11/30/karacs...,www.telex.hu,Karácsony benyújtotta a rendkívüli közgyűlésre...,Megjelent a főváros honlapján Karácsony Gergel...,Thüringer Barbara,2025-11-30 00:00:00,"[előterjesztés, Fővárosi Önkormányzat, Belföld...",NaN
1,https://www.telex.hu/after/2025/11/28/sex-pist...,www.telex.hu,Budapestre jön a Sex Pistols frontemberének ze...,"A Sex Pistols egykori frontemberének, Johnny R...",Dévai László,2025-11-28 00:00:00,"[Sex Pistols, Johnny Rotten, Public Image Ltd,...",NaN
2,https://www.telex.hu/pr-cikk/2025/11/28/igy-sz...,www.telex.hu,Így szerezhet most 110 eurótól tulajdonrészt a...,Kihagyhatatlan lehetőséget kínál a kisbefektet...,Ez itt egy PR-cikk,2025-11-28 00:00:00,"[PR2, PR-cikk]",NaN
3,https://www.telex.hu/eszkombajn/2025/10/06/szt...,www.telex.hu,Amikor Sztálin cikkeket írt a Szabad Népbe,"Egy szókimondó, ellentmondást nem tűrő új szer...",Ághassi Attila,2025-10-06 00:00:00,"[Szabad Nép, Észkombájn, Sztálin]",NaN
4,https://www.telex.hu/belfold/2025/11/30/dohany...,www.telex.hu,Lezuhant egy kőkorlát darabja egy Dohány utcai...,Egy olvasónk szerint szombat este a budapesti ...,Thüringer Barbara,2025-11-30 00:00:00,"[Dohány utca, Belföld, volkswagen, rendőrség, ...",NaN
...,...,...,...,...,...,...,...,...
971,https://papageno.hu/blogok/mozart-planet/2025/...,papageno.hu,"Paul Lewis: „Egy zongora akkor a legjobb, ha é...",Paul Lewis a kortárs zenei élet egyik legelism...,Ur Máté,2025-11-28 04:02:25,"[paul lewis, concerto budapest, beethoven, int...",NaN
979,https://papageno.hu/kiallitas/2025/11/a-magyar...,papageno.hu,A magyar neoavantgárd mestere - Keserü Ilona,Keserü Ilona (1933) a hazai kortárs képzőművés...,Papageno,2025-11-29 04:28:54,"[képzőművészet, keserü ilona]",NaN
981,https://papageno.hu/promocio/2025/11/a-kekszak...,papageno.hu,Stuttgartban és Bukaresten is sikert aratott A...,2025-ben két alkalommal adta elő Bartók A kéks...,Papageno,2025-11-25 18:29:34,"[kamara, cser péter, hajdú diána, dénes istván...",NaN
984,https://papageno.hu/magazin/2025/10/lapozzon-b...,papageno.hu,Lapozzon bele a Papageno novemberi számába!,Megjelent a Papageno magazin novemberi száma. ...,Papageno,2025-10-21 10:13:47,[papageno magazin 2025],NaN


In [17]:
import pandas as pd
import re

def clean_list_column(lst, mode="authors"):
    """Clean a list of strings depending on column mode."""
    if not isinstance(lst, list):
        return lst  # leave non-lists untouched

    cleaned = []
    for item in lst:
        if not isinstance(item, str):
            continue
        
        s = item.strip()

        if mode == "authors":
            # remove 'Szerző' or 'Szerzo' prefix (accent-insensitive)
            s = re.sub(r"^(szerz[oő]i?|szerző)\s*[:\-]*\s*", "", s, flags=re.IGNORECASE)

        elif mode == "keywords":
            # remove leading hashtag
            s = re.sub(r"^#\s*", "", s)
            # replace underscores with spaces
            s = s.replace("_", " ")
            # lowercase
            s = s.lower()
            # normalize extra spaces
            s = re.sub(r"\s+", " ", s).strip()

        cleaned.append(s)

    return cleaned


def clean_dataframe(df):
    """Clean 'authors' and 'keywords' columns in-place."""
    if "authors" in df.columns:
        df["authors"] = df["authors"].apply(lambda x: clean_list_column(x, mode="authors"))

    if "keywords" in df.columns:
        df["keywords"] = df["keywords"].apply(lambda x: clean_list_column(x, mode="keywords"))

    return df

In [19]:
df =clean_dataframe(date_df)

In [22]:
df.dtypes



url                 object
page                object
title               object
text                object
author              object
date        datetime64[ns]
keywords            object
error               object
dtype: object